# Alpha Research 2026 — Institutional Systematic Macro Pipeline

## Six-Strategy Pod: PDRRM · TPMCR · MAERM · ISRC · VSRA · FDSP

> **Architecture**: Python 3.13 orchestration layer → C++26 signal engines (nanobind bridge) → Polars data pipeline → KPI reporting

### Strategy Universe
| # | Strategy | Asset Class | Edge |
|---|----------|-------------|------|
| 1 | **PDRRM** | G10 FX Futures (6J,6E,6B…) | Real rate differential momentum × CB divergence |
| 2 | **TPMCR** | Rates Futures (ZN,ZB,RX,G) | ACM term premium lag + HMM curve regime |
| 3 | **MAERM** | Equity Index Futures (ES,NQ…) | EPS revision × ISM interaction |
| 4 | **ISRC** | Energy Futures (CL,NG,RB) | EIA inventory surprise × roll structure |
| 5 | **VSRA** | SPX Options + VIX Futures | VRP + term structure carry + skew anomaly |
| 6 | **FDSP** | Cross-Asset (UST+USD+GLD+EQ) | Fiscal shock propagation kernels |

**KPI Targets**: Sharpe > 1.0 · IC > 4% · ICIR > 0.5 · MDD < 15% · Net Alpha > 100 bps/yr

In [ ]:
# ── Environment setup ─────────────────────────────────────────────────────────
import sys, os
from pathlib import Path

# Allow importing alpha_research without absolute paths
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(repo_root / 'src' / 'python'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

print(f'Python {sys.version}')
print(f'NumPy  {np.__version__}')
print(f'Polars {pl.__version__}')

# Try to load C++ engine
try:
    import alpha_cpp
    print(f'alpha_cpp C++26 engine: ✅ loaded')
except ImportError:
    print('alpha_cpp C++26 engine: ⚠️  not built — pure-Python fallback active')
    print('  Build with: cmake --build cmake-build --target alpha_cpp')

## 1. Data Generation & Inspection

We generate 10 years (2520 trading days) of synthetic G10 macro data calibrated to 2015-2025 statistics.

In [ ]:
from alpha_research.data import (
    generate_synthetic_data, extract_numpy_panels,
    G10_FX_SYMBOLS, ENERGY_SYMBOLS, CROSS_ASSET_SYMBOLS
)

N_DAYS, N_CCY = 2520, 8
raw = generate_synthetic_data(n_days=N_DAYS, n_currencies=N_CCY, seed=42)
panels = extract_numpy_panels(raw, n_currencies=N_CCY)

dates = pl.date_range(start=pl.date(2015,1,2), end=None, interval='1bd', eager=True).head(N_DAYS)

print('Data shapes:')
for k, v in panels.items():
    print(f'  {k:20s}: {v.shape}')

In [ ]:
# ── Real Rate Differentials (PDRRM foundation) ───────────────────────────────
real_rates = panels['nominal_rates'] - panels['breakevens']
# Differential vs USD (col 0)
rrd = real_rates - real_rates[:, [0]]

dates_np = dates.to_numpy()

fig = go.Figure()
for i, ccy in enumerate(G10_FX_SYMBOLS[:N_CCY]):
    fig.add_trace(go.Scatter(
        x=dates_np, y=rrd[:, i]*100,
        name=ccy, line=dict(width=1.5),
        visible=True if ccy in ('6J','6E','6B') else 'legendonly'
    ))

fig.update_layout(
    title='Real Rate Differential vs USD (%) — PDRRM Signal Foundation',
    xaxis_title='Date', yaxis_title='Real Rate Diff (%)',
    template='plotly_dark', height=420,
    legend=dict(orientation='h', y=-0.2)
)
fig.show()

## 2. PDRRM Signal Computation (C++26 Engine)

The PDRRM engine computes three orthogonal components via the C++26 `PDRRMEngine` and combines them with ridge-estimated weights.

In [ ]:
from alpha_research.signals import compute_pdrrm_signals

pdrrm_sigs, pdrrm_pos = compute_pdrrm_signals(
    nominal_rates  = panels['nominal_rates'],
    breakevens     = panels['breakevens'],
    cb_surprises   = panels['cb_surprises'],
    forward_premia = panels['forward_premia'],
    fx_returns     = panels['fx_returns'],
    ridge_lambda   = 0.10,
    train_fraction = 0.60,
)
print(f'PDRRM signals: {pdrrm_sigs.shape}  positions: {pdrrm_pos.shape}')
print(f'Signal stats  — mean: {pdrrm_sigs[300:].mean():.4f}  std: {pdrrm_sigs[300:].std():.4f}')
print(f'Position stats — mean: {np.abs(pdrrm_pos[300:]).mean():.4f}  max: {np.abs(pdrrm_pos).max():.4f}')

In [ ]:
# ── PDRRM signal heatmap (cross-section × time) ───────────────────────────────
fig = go.Figure(data=go.Heatmap(
    z=pdrrm_sigs[300:].T,
    x=dates_np[300:],
    y=list(G10_FX_SYMBOLS[:N_CCY]),
    colorscale='RdBu', zmid=0,
    colorbar=dict(title='z-score')
))
fig.update_layout(
    title='PDRRM Combined Signal — Cross-Section × Time',
    xaxis_title='Date', yaxis_title='Currency',
    template='plotly_dark', height=380
)
fig.show()

## 3. Full Six-Strategy Pipeline

The master signal orchestrator routes data to all six C++26 engines and returns `SignalBundle` objects with signals, positions, and KPIs.

In [ ]:
from alpha_research.signals import compute_master_signal

bundles = compute_master_signal(panels, dates)
print('Strategies computed:', list(bundles.keys()))

## 4. KPI Dashboard

All KPIs are computed by the C++26 `BacktestEngine` — Sharpe, Sortino, Calmar, CAGR, MDD, IC, ICIR, half-life decay.

In [ ]:
from alpha_research.backtest import BacktestOrchestrator

orc = BacktestOrchestrator(n_days=N_DAYS, n_currencies=N_CCY)
report = orc.run()
print(report)

In [ ]:
# ── KPI spider / radar chart ─────────────────────────────────────────────────
if 'sharpe' in report.columns:
    strategies = report['strategy'].to_list()
    sharpes    = [float(v) for v in report['sharpe'].to_list()]
    mean_ics   = [float(v)*100 for v in report.get_column('mean_ic').to_list()] if 'mean_ic' in report.columns else [0]*len(strategies)

    fig = make_subplots(rows=1, cols=2,
        subplot_titles=['Sharpe Ratios by Strategy', 'Mean IC (%) by Strategy'])

    colors = px.colors.qualitative.Plotly[:len(strategies)]
    fig.add_trace(go.Bar(x=strategies, y=sharpes, marker_color=colors,
                         text=[f'{v:.2f}' for v in sharpes], textposition='outside',
                         name='Sharpe'), row=1, col=1)
    fig.add_trace(go.Bar(x=strategies, y=mean_ics, marker_color=colors,
                         text=[f'{v:.2f}%' for v in mean_ics], textposition='outside',
                         name='IC%', showlegend=False), row=1, col=2)

    fig.add_hline(y=1.0, line_dash='dash', line_color='lime',
                  annotation_text='Target Sharpe=1.0', row=1, col=1)
    fig.add_hline(y=4.0, line_dash='dash', line_color='lime',
                  annotation_text='Target IC=4%', row=1, col=2)

    fig.update_layout(template='plotly_dark', height=420,
                      title='Strategy KPI Dashboard')
    fig.show()

## 5. PDRRM Cumulative PnL & Drawdown

Vol-targeted positions generate daily PnL. We compute and plot the cumulative equity curve and underwater plot.

In [ ]:
# Daily PnL: positions × next-day returns
fx_ret = panels['fx_returns']
daily_pnl = np.zeros(N_DAYS)
for t in range(21, N_DAYS - 1):
    daily_pnl[t] = np.dot(pdrrm_pos[t], fx_ret[t+1])

cum_pnl    = np.cumsum(daily_pnl)
running_max= np.maximum.accumulate(cum_pnl)
drawdown   = (running_max - cum_pnl) / (np.abs(running_max) + 1e-8)

ann_vol    = np.std(daily_pnl[21:]) * np.sqrt(252)
ann_return = np.mean(daily_pnl[21:]) * 252
sharpe     = ann_return / (ann_vol + 1e-8)
mdd        = drawdown.max()
print(f'PDRRM  Sharpe={sharpe:.3f}  Ann.Return={ann_return*100:.2f}%  MDD={mdd*100:.2f}%')

fig = make_subplots(rows=2, cols=1, row_heights=[0.65, 0.35],
                    shared_xaxes=True, vertical_spacing=0.05,
                    subplot_titles=['Cumulative PnL (PDRRM)', 'Drawdown'])

fig.add_trace(go.Scatter(x=dates_np[21:], y=cum_pnl[21:]*100,
    fill='tozeroy', line=dict(color='#00C853', width=1.5), name='Cum PnL %'), row=1, col=1)
fig.add_trace(go.Scatter(x=dates_np[21:], y=-drawdown[21:]*100,
    fill='tozeroy', fillcolor='rgba(211,47,47,0.25)',
    line=dict(color='#D32F2F', width=1), name='Drawdown %'), row=2, col=1)

fig.update_yaxes(title_text='PnL (%)', row=1, col=1)
fig.update_yaxes(title_text='DD (%)', row=2, col=1)
fig.update_layout(template='plotly_dark', height=550,
    title=f'PDRRM Equity Curve | Sharpe={sharpe:.2f} | MDD={mdd*100:.1f}%')
fig.show()

## 6. Information Coefficient (IC) Analysis

Rolling IC with 60-day window. IC > 4% with ICIR > 0.5 is the production target.

In [ ]:
from scipy.stats import spearmanr

rolling_ic = []
window_ic  = 60
for t in range(window_ic, N_DAYS - 1):
    s = pdrrm_sigs[t]
    r = fx_ret[t+1]
    ic, _ = spearmanr(s, r)
    rolling_ic.append(ic if not np.isnan(ic) else 0.0)

rolling_ic_arr = np.array(rolling_ic)
dates_ic = dates_np[window_ic: N_DAYS - 1]
mean_ic = rolling_ic_arr.mean()
icir    = mean_ic / (rolling_ic_arr.std() + 1e-8)

# Half-life from C++ if available, else AR(1)
try:
    import alpha_cpp
    hl = alpha_cpp.half_life_decay(rolling_ic_arr.astype(np.float64))
except Exception:
    from numpy.polynomial import polynomial as P
    hl = float('inf')

print(f'Mean IC={mean_ic:.4f}  ICIR={icir:.3f}  HL={hl:.1f}d')

fig = go.Figure()
fig.add_trace(go.Scatter(x=dates_ic, y=rolling_ic_arr*100,
    line=dict(color='#40C4FF', width=1), name='Daily IC (%)'))
fig.add_trace(go.Scatter(x=dates_ic,
    y=pl.Series(rolling_ic_arr*100).rolling_mean(60).to_numpy(),
    line=dict(color='#FF6F00', width=2), name='60d Rolling Mean IC (%)'))
fig.add_hline(y=4.0, line_dash='dash', line_color='lime', annotation_text='Target IC=4%')
fig.add_hline(y=0.0, line_color='white', line_width=0.5)
fig.update_layout(template='plotly_dark', height=380,
    title=f'PDRRM Rolling IC | Mean={mean_ic*100:.2f}%  ICIR={icir:.2f}',
    xaxis_title='Date', yaxis_title='IC (%)')
fig.show()

## 7. Black-Swan Stress Tests

We measure strategy PnL and drawdown during historical stress episodes:
- **Mar 2020** — COVID-19 crash (VIX → 85)
- **2022** — Fed hiking shock (fastest rate cycle in 40 years)
- **Aug 2024** — JPY carry unwind (Nikkei -12% in one day)

In [ ]:
from alpha_research.backtest import BLACK_SWAN_WINDOWS

stress_rows = []
for event, (t0, t1) in BLACK_SWAN_WINDOWS.items():
    t1 = min(t1, N_DAYS - 1)
    episode_pnl   = daily_pnl[t0:t1].sum() * 100
    episode_vol   = daily_pnl[t0:t1].std() * np.sqrt(252) * 100
    episode_mdd   = (np.maximum.accumulate(np.cumsum(daily_pnl[t0:t1])) -
                     np.cumsum(daily_pnl[t0:t1])).max() * 100
    stress_rows.append({
        'Event': event, 'Days': t1-t0,
        'PnL_%': round(episode_pnl,2),
        'Vol_%': round(episode_vol,2),
        'MDD_%': round(episode_mdd,2),
    })

stress_df = pl.DataFrame(stress_rows)
print(stress_df)

# Visualise
fig = go.Figure(data=go.Table(
    header=dict(values=list(stress_df.columns),
                fill_color='#1565C0', font=dict(color='white', size=12), align='left'),
    cells=dict(values=[stress_df[c].to_list() for c in stress_df.columns],
               fill_color=[['#0D1117','#161B22']*10],
               font=dict(color='white', size=11), align='left')
))
fig.update_layout(template='plotly_dark', title='Black-Swan Stress Test Results',
                  height=300)
fig.show()

## 8. Signal Decay Monitor

Production C++ `SignalDecayMonitor` tracks rolling IC mean and half-life. Alert fires if IC half-life < 30 days or rolling mean < 2%.

In [ ]:
from alpha_research.signals import DecayMonitor

dm = DecayMonitor('PDRRM')
alerts = []
for ic in rolling_ic_arr:
    alerts.append(dm.update(ic))

n_alerts = sum(alerts)
hl = dm.half_life()
mean_ic_dm = dm.half_life()
print(f'Decay Monitor — Decay alerts triggered: {n_alerts}')
print(f'Current half-life: {hl:.1f} days')
print(f'Status: {"⚠️  DECAY DETECTED" if dm.alert() else "✅ Signal healthy"}')

fig = go.Figure()
fig.add_trace(go.Scatter(x=dates_ic, y=rolling_ic_arr*100,
    line=dict(color='#40C4FF', width=1), name='IC (%)'))
alert_dates = [dates_ic[i] for i, a in enumerate(alerts) if a]
alert_vals  = [rolling_ic_arr[i]*100 for i, a in enumerate(alerts) if a]
fig.add_trace(go.Scatter(x=alert_dates, y=alert_vals,
    mode='markers', marker=dict(symbol='x', size=10, color='red'),
    name='Decay Alert'))
fig.add_hline(y=2.0, line_dash='dot', line_color='orange',
    annotation_text='IC decay threshold (2%)')
fig.update_layout(template='plotly_dark', height=380,
    title=f'Signal Decay Monitor — HL={hl:.1f}d | Alerts={n_alerts}')
fig.show()

## 9. Term Structure & Volatility Surface (TPMCR + VSRA)

Visualise the ACM term premium trend and VIX term structure used by TPMCR and VSRA engines.

In [ ]:
fig = make_subplots(rows=2, cols=2,
    subplot_titles=['ACM Term Premium (bps) — TPMCR Input',
                    'Yield Curve: 2Y vs 10Y vs 30Y',
                    'VIX Spot — VSRA Regime',
                    'Volatility Risk Premium (IV - RV)'],
    vertical_spacing=0.12, horizontal_spacing=0.08)

# ACM Term Premium
fig.add_trace(go.Scatter(x=dates_np, y=panels['tp_acm']*100,
    line=dict(color='#FF9800'), name='TP_ACM'), row=1, col=1)

# Yield curve
for label, col, color in [('2Y', 'yield_2y','#40C4FF'),
                            ('10Y','yield_10y','#FFEB3B'),
                            ('30Y','yield_30y','#FF5722')]:
    fig.add_trace(go.Scatter(x=dates_np, y=panels[col]*100,
        line=dict(color=color, width=1.2), name=label,
        legendgroup='yields'), row=1, col=2)

# VIX
fig.add_trace(go.Scatter(x=dates_np, y=panels['vix_spot'],
    fill='tozeroy', fillcolor='rgba(211,47,47,0.15)',
    line=dict(color='#D32F2F'), name='VIX'), row=2, col=1)
fig.add_hline(y=20, line_dash='dash', line_color='grey', row=2, col=1)

# VRP
vrp = (panels['iv_atm'] - panels['rv_21d']) * 100
fig.add_trace(go.Scatter(x=dates_np, y=vrp,
    fill='tozeroy', line=dict(color='#7C4DFF'), name='VRP (%)'), row=2, col=2)
fig.add_hline(y=0, line_color='white', line_width=0.5, row=2, col=2)

fig.update_layout(template='plotly_dark', height=650,
    title='Rates & Volatility Surface Dashboard')
fig.show()

## 10. Master Portfolio: Cross-Signal Correlation & PCA Risk

PCA decomposition of the combined signal matrix to verify no double-loading on PC1 (risk-on/off), PC2 (rates level), PC3 (vol regime).

In [ ]:
from sklearn.decomposition import PCA

# Stack all strategy daily PnLs into a combined matrix [T x 6]
strategy_pnls = {}
for name, bundle in bundles.items():
    pos = bundle.positions
    # Use proxy returns per strategy
    if name == 'PDRRM':
        ret_mat = panels['fx_returns'][:, :pos.shape[1]]
    elif name == 'ISRC':
        ret_mat = panels['energy_returns'][:, :pos.shape[1]]
    elif name == 'FDSP':
        ret_mat = panels['ca_returns'][:, :pos.shape[1]]
    else:
        ret_mat = np.zeros_like(pos)
    pnl_series = (pos * ret_mat).sum(axis=1)
    strategy_pnls[name] = pnl_series

pnl_matrix = np.column_stack(list(strategy_pnls.values()))[21:]
strat_names = list(strategy_pnls.keys())

# Correlation matrix
corr = np.corrcoef(pnl_matrix.T)

fig = go.Figure(data=go.Heatmap(
    z=corr, x=strat_names, y=strat_names,
    colorscale='RdBu', zmid=0, zmin=-1, zmax=1,
    text=np.round(corr, 2), texttemplate='%{text}',
    colorbar=dict(title='ρ')
))
fig.update_layout(template='plotly_dark', height=460,
    title='Cross-Strategy PnL Correlation Matrix')
fig.show()

# PCA explained variance
pca = PCA(n_components=min(6, len(strat_names)))
pca.fit(pnl_matrix)
ev = pca.explained_variance_ratio_ * 100
print('PCA Explained Variance:')
for i, v in enumerate(ev):
    print(f'  PC{i+1}: {v:.1f}%')

In [ ]:
# ── Master combined PnL ──────────────────────────────────────────────────────
from alpha_research.signals import MASTER_WEIGHTS

master_pnl = np.zeros(len(pnl_matrix))
for i, name in enumerate(strat_names):
    w = MASTER_WEIGHTS.get(name, 1/6)
    master_pnl += w * pnl_matrix[:, i]

cum_master    = np.cumsum(master_pnl)
master_sharpe = master_pnl.mean()*252 / (master_pnl.std()*np.sqrt(252) + 1e-8)
master_mdd    = (np.maximum.accumulate(cum_master) - cum_master).max()
dates_master  = dates_np[21:]

print(f'Master Portfolio — Sharpe={master_sharpe:.3f}  MDD={master_mdd*100:.2f}%')

fig = go.Figure()
fig.add_trace(go.Scatter(x=dates_master, y=cum_master*100,
    fill='tozeroy', line=dict(color='#00E676', width=2),
    name='Master Portfolio'))
for name in strat_names:
    i = list(strat_names).index(name)
    fig.add_trace(go.Scatter(x=dates_master,
        y=np.cumsum(pnl_matrix[:, i])*100,
        line=dict(width=0.8), name=name, visible='legendonly'))

fig.update_layout(template='plotly_dark', height=500,
    title=f'Master Portfolio Cumulative PnL | Sharpe={master_sharpe:.2f} | MDD={master_mdd*100:.1f}%',
    xaxis_title='Date', yaxis_title='Cum PnL (%)',
    legend=dict(orientation='h', y=-0.2))
fig.show()

## 11. Full KPI Summary Table

| Metric | Target | Description |
|--------|--------|-------------|
| **Sharpe** | > 1.0 | Ann. return / ann. vol |
| **Sortino** | > 1.5 | Ann. return / downside vol |
| **Calmar** | > 0.5 | CAGR / MDD |
| **Mean IC** | > 4% | Spearman signal-return rank correlation |
| **ICIR** | > 0.5 | Mean IC / std(IC) — signal consistency |
| **MDD** | < 15% | Maximum drawdown from peak |
| **HL (days)** | > 60 | IC half-life — signal longevity |

In [ ]:
print('=== FULL KPI REPORT ===')
print(report)
print()
print(f'=== MASTER PORTFOLIO ===')
print(f'  Sharpe:        {master_sharpe:.3f}')
print(f'  MDD:           {master_mdd*100:.2f}%')
print(f'  Ann. PnL:      {master_pnl.mean()*252*100:.2f}%')
print(f'  Ann. Vol:      {master_pnl.std()*np.sqrt(252)*100:.2f}%')